The Movies Dataset i MovieLens łączymy za pomocą links.csv

In [8]:
import pandas as pd
from pathlib import Path

archive_path = Path("data/archive")
small_path = Path("data/ml-latest-small")

def safe_int64(col):
    return pd.to_numeric(col, errors='coerce').astype('Int64')

def load_cleaned_files(path):
    files = list(path.glob("*_cleaned.csv"))
    dfs = {}
    for f in files:
        name = f.stem.replace("_cleaned", "")
        dfs[name] = pd.read_csv(f, low_memory=False)
    return dfs

archive_dfs = load_cleaned_files(archive_path)
small_dfs = load_cleaned_files(small_path)

if 'links' in archive_dfs and 'movies_metadata' in archive_dfs:
    archive_dfs['links']['tmdbId'] = safe_int64(archive_dfs['links']['tmdbId'])
    archive_dfs['movies_metadata']['id'] = safe_int64(archive_dfs['movies_metadata']['id'])
    archive_dfs['movies_full'] = archive_dfs['movies_metadata'].merge(
        archive_dfs['links'][['tmdbId', 'imdbId']],
        left_on='id', right_on='tmdbId', how='left'
    )

if 'links' in small_dfs and 'movies' in small_dfs:
    small_dfs['links']['tmdbId'] = safe_int64(small_dfs['links']['tmdbId'])
    small_dfs['movies']['movieId'] = safe_int64(small_dfs['movies']['movieId'])
    small_dfs['movies_full'] = small_dfs['movies'].merge(
        small_dfs['links'][['tmdbId', 'imdbId']],
        left_on='movieId', right_on='tmdbId', how='left'
    )

print(archive_dfs['movies_full'].head())
print(small_dfs['movies_full'].head())

   adult           belongs_to_collection    budget                    genres  \
0  False            toy_story_collection  30000000   animation comedy family   
1  False                             NaN  65000000  adventure fantasy family   
2  False       grumpy_old_men_collection         0            romance comedy   
3  False                             NaN  16000000      comedy drama romance   
4  False  father_of_the_bride_collection         0                    comedy   

                               homepage     id    imdb_id original_language  \
0  http://toystory.disney.com/toy-story    862  tt0114709                en   
1                               unknown   8844  tt0113497                en   
2                               unknown  15602  tt0113228                en   
3                               unknown  31357  tt0114885                en   
4                               unknown  11862  tt0113041                en   

                original_title  \
0         

In [9]:
import pandas as pd
from pathlib import Path

data_path = Path("data")

ml_path = data_path / "ml-latest-small"
tmdb_path = data_path / "archive"

def safe_int64(col):
    return pd.to_numeric(col, errors="coerce").astype("Int64")


ratings = pd.read_csv(ml_path / "ratings_cleaned.csv")
movies = pd.read_csv(ml_path / "movies_cleaned.csv")
links = pd.read_csv(ml_path / "links_cleaned.csv")
tags = pd.read_csv(ml_path / "tags_cleaned.csv")

movies_metadata = pd.read_csv(tmdb_path / "movies_metadata_cleaned.csv", low_memory=False)
credits = pd.read_csv(tmdb_path / "credits_cleaned.csv")
keywords = pd.read_csv(tmdb_path / "keywords_cleaned.csv")

links["tmdbId"] = safe_int64(links["tmdbId"])
links["movieId"] = safe_int64(links["movieId"])

movies_metadata["id"] = safe_int64(movies_metadata["id"])
credits["id"] = safe_int64(credits["id"])
keywords["id"] = safe_int64(keywords["id"])

credits["cast"] = credits["cast_main"]
credits["director"] = credits["director"]
credits["writers"] = credits["writers"]

keywords["keywords"] = keywords["keywords_clean"]

tmdb = movies_metadata.merge(
    credits[["id", "cast", "director", "writers"]],
    on="id",
    how="left"
)

tmdb = tmdb.merge(
    keywords[["id", "keywords"]],
    on="id",
    how="left"
)
tmdb = tmdb.sort_values("popularity", ascending=False)
tmdb = tmdb.drop_duplicates(subset=["id"])

ml = ratings.merge(movies, on="movieId", how="left")
ml = ml.merge(links, on="movieId", how="left")


ml = ml.dropna(subset=["tmdbId"]) 
full_dataset = ml.merge(
    tmdb,
    left_on="tmdbId",
    right_on="id",
    how="left"
)

tags_agg = tags.groupby("movieId")["tag"].apply(list).reset_index()
full_dataset = full_dataset.merge(tags_agg, on="movieId", how="left")
if "release_date" in full_dataset.columns:
    full_dataset["release_year"] = pd.to_datetime(
        full_dataset["release_date"],
        errors="coerce"
    ).dt.year

final_data = full_dataset[[
    "userId",
    "movieId",
    "rating",
    "title_x",
    "genres_x",
    "tmdbId",
    "budget",
    "revenue",
    "runtime",
    "popularity",
    "vote_average",
    "cast",
    "director",
    "writers",
    "keywords",
    "tag",
    "release_year",
]].copy()

#
if "production_countries" in full_dataset.columns:
    final_data["production_countries"] = full_dataset["production_countries"]
else:
    final_data["production_countries"] = pd.NA

final_data["production_countries"] = final_data["production_countries"].fillna("")
final_data["primary_production_country"] = final_data["production_countries"].apply(
    lambda x: x.split()[0] if isinstance(x, str) and x.strip() != "" else pd.NA
).astype("string")

final_data["release_year"] = final_data["release_year"].astype("Int64")
final_data["budget"] = final_data["budget"].replace(0, pd.NA)
final_data["revenue"] = final_data["revenue"].replace(0, pd.NA)

final_data.to_csv("data/final_dataset.csv", index=False)


In [10]:
from pathlib import Path
import pandas as pd

fp = Path("data/final_dataset.csv")
if fp.exists():
    df = pd.read_csv(fp, low_memory=False)
    before = len(df)
    df = df.dropna(subset=["budget", "revenue"])
    after = len(df)
    print(f"Usunięto {before-after} wierszy z brakującym budget lub revenue")
    df.to_csv(fp, index=False)
else:
    print("Plik data/final_dataset.csv nie istnieje. Uruchom najpierw komórkę generującą go.")


Usunięto 20952 wierszy z brakującym budget lub revenue


In [11]:
final_data.duplicated(subset=['userId','movieId']).sum()

np.int64(0)

In [12]:
print(final_data.shape)
print(ratings.shape)

(100823, 19)
(100836, 6)


In [13]:
tmdb.duplicated(subset=["id"]).sum()

np.int64(0)

In [14]:
len(final_data)


100823

In [15]:
print(ratings.shape[0])

100836
